# LogReg v4 — Model Analysis

Minimal, detailed audit of the current production model (`models/logreg_v4`).

**What we check:**
1. Dataset shape, label balance, slot coverage
2. Scaled coefficient importance
3. Discrimination (AUC, accuracy, confusion)
4. Calibration (reliability curve, Brier raw vs calibrated)
5. Edge vs realized outcome (is `p_up - 0.5` monotonic in win rate?)
6. Slot-level vs row-level accuracy gap

Target: `y = 1 if Up resolves else 0` on 5-minute BTC windows.

In [ ]:
print(1)

In [ ]:
import json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    brier_score_loss, log_loss, roc_auc_score,
    accuracy_score, confusion_matrix,
)
from sklearn.calibration import calibration_curve

MODEL_DIR = Path('../models/logreg_v4')
meta = json.loads((MODEL_DIR / 'logreg_meta.json').read_text())
model = pickle.load(open(MODEL_DIR / 'logreg_model.pkl', 'rb'))
scaler = pickle.load(open(MODEL_DIR / 'logreg_scaler.pkl', 'rb'))
calibrator = pickle.load(open(MODEL_DIR / 'logreg_calibrator.pkl', 'rb'))
features = meta['features']
df = pd.read_csv(MODEL_DIR / 'training_data.csv')
print(f"rows={len(df)}  slots={df['slot_ts'].nunique()}  features={len(features)}")
print(f"meta reports: n_train_rows={meta['n_train_rows']}, n_valid_rows={meta['n_valid_rows']}")

## 1. Dataset shape & label balance

In [ ]:
base_rate = df['y'].mean()
slot_rate = df.groupby('slot_ts')['y'].first().mean()
print(f"row-level P(Up)  = {base_rate:.4f}")
print(f"slot-level P(Up) = {slot_rate:.4f}")
print(f"rows per slot    = {len(df) / df['slot_ts'].nunique():.1f}")
df[['time_to_expiry'] + [f for f in features if f != 'time_to_expiry'][:4]].describe().round(5)

## 2. Scaled coefficient importance

Since the logistic regression operates on standard-scaled features, `|coef|` is directly comparable across features.

In [ ]:
coef = np.asarray(meta['coef'])
imp = pd.DataFrame({'feature': features[:len(coef)], 'coef': coef})
imp['abs'] = imp['coef'].abs()
imp = imp.sort_values('abs', ascending=True)
fig, ax = plt.subplots(figsize=(6, 5))
colors = ['#d62728' if c < 0 else '#2ca02c' for c in imp['coef']]
ax.barh(imp['feature'], imp['coef'], color=colors)
ax.axvline(0, color='k', lw=0.5)
ax.set_title(f"Scaled coefficients (intercept={meta['intercept']:.3f})")
plt.tight_layout(); plt.show()
imp.sort_values('abs', ascending=False).head(8).round(4)

## 3. Score the full dataset
Note: this mixes train+valid, so metrics here are optimistic vs the held-out numbers in `meta`.

In [ ]:
X = df[features[:len(coef)]].values
y = df['y'].values
Xs = scaler.transform(X)
# Sidestep sklearn version mismatch (old pickle may be missing attrs like
# `multi_class` that newer sklearn's predict_proba requires) by computing
# p = sigmoid(Xs @ coef + intercept) directly from the fitted params.
w = np.asarray(model.coef_).ravel()
b = float(np.asarray(model.intercept_).ravel()[0])
p_raw = 1.0 / (1.0 + np.exp(-(Xs @ w + b)))

if hasattr(calibrator, 'predict'):
    p_cal = calibrator.predict(p_raw)
elif hasattr(calibrator, 'transform'):
    p_cal = calibrator.transform(p_raw)
else:
    p_cal = np.asarray(calibrator(p_raw))

def metrics(p, label):
    return {
        'label': label,
        'brier': brier_score_loss(y, p),
        'logloss': log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)),
        'auc': roc_auc_score(y, p),
        'acc@0.5': accuracy_score(y, p > 0.5),
    }
pd.DataFrame([metrics(p_raw, 'raw'), metrics(p_cal, 'calibrated')]).round(4)

In [ ]:
cm = confusion_matrix(y, p_cal > 0.5)
tn, fp, fn, tp = cm.ravel()
print(f"            pred_dn  pred_up")
print(f"true_dn     {tn:6d}   {fp:6d}")
print(f"true_up     {fn:6d}   {tp:6d}")
print(f"precision(up)={tp/(tp+fp):.3f}  recall(up)={tp/(tp+fn):.3f}")

## 4. Calibration — is P(Up) trustworthy?
The calibrated curve should hug the diagonal. Deviation means sizing via Kelly is biased.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for p, name, ax in [(p_raw, 'raw', axes[0]), (p_cal, 'calibrated', axes[1])]:
    frac_pos, mean_pred = calibration_curve(y, p, n_bins=10, strategy='quantile')
    ax.plot([0, 1], [0, 1], 'k--', lw=0.5)
    ax.plot(mean_pred, frac_pos, 'o-')
    ax.set(xlabel='predicted P(Up)', ylabel='empirical P(Up)',
           title=f'{name}  brier={brier_score_loss(y, p):.4f}', xlim=(0,1), ylim=(0,1))
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(p_cal[y == 1], bins=40, alpha=0.6, label='y=Up', color='#2ca02c')
ax.hist(p_cal[y == 0], bins=40, alpha=0.6, label='y=Down', color='#d62728')
ax.axvline(0.5, color='k', lw=0.5)
ax.set(xlabel='calibrated p_up', ylabel='count', title='Score distribution by outcome')
ax.legend(); plt.tight_layout(); plt.show()

## 5. Edge vs realized win rate
Bucket by model edge (`|p_up - 0.5|`) — larger edge should yield higher accuracy. If it's flat, the model has no usable signal even when it's "confident".

In [ ]:
edge = np.abs(p_cal - 0.5)
picked = (p_cal > 0.5).astype(int)
correct = (picked == y).astype(int)
buckets = pd.qcut(edge, 10, duplicates='drop')
tbl = pd.DataFrame({'edge': edge, 'correct': correct, 'bucket': buckets}) \
        .groupby('bucket', observed=True) \
        .agg(n=('correct', 'size'), mean_edge=('edge', 'mean'), win_rate=('correct', 'mean'))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(tbl['mean_edge'], tbl['win_rate'], 'o-')
ax.axhline(0.5, color='k', lw=0.5)
ax.set(xlabel='|p_up - 0.5|', ylabel='win rate', title='Edge vs realized accuracy')
plt.tight_layout(); plt.show()
tbl.round(4)

## 6. Slot-level accuracy
A slot (5-min window) is "won" if the *majority* of its rows point the right way. This is what matters for trading — rows inside the same slot are highly correlated.

In [ ]:
df_s = df.assign(p=p_cal, pick=(p_cal > 0.5).astype(int))
slot = df_s.groupby('slot_ts').agg(
    y=('y', 'first'),
    p_mean=('p', 'mean'),
    pick_mean=('pick', 'mean'),
)
slot['pick'] = (slot['p_mean'] > 0.5).astype(int)
slot_acc = (slot['pick'] == slot['y']).mean()
print(f"row-level   acc = {(df_s['pick'] == df_s['y']).mean():.4f}")
print(f"slot-level  acc = {slot_acc:.4f}")
print(f"meta reports    = {meta['slot_accuracy']:.4f}  (held-out valid)")

## Takeaways

- **Top drivers** (scaled coef magnitude) — see section 2. `ret_180s` dominates; short-horizon returns partially cancel.
- **Calibration** matters more than accuracy here: Kelly sizing multiplies the calibration bias into dollar loss.
- **Slot vs row** accuracy gap: if slot accuracy is notably higher, a single decision per slot (or majority vote) is the right trading unit.
- **Edge-vs-win-rate** monotonicity is the sharpest check that the model's confidence is informative at all.

Next step if any of the above look off: refit with walk-forward splits by `slot_ts`, not random rows, and re-check calibration on the true held-out tail.